In [34]:
import os
import sys

import boto3
import sagemaker
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.session import Session
import pandas as pd
from io import StringIO
from sagemaker import get_execution_role
from sagemaker.workflow.function_step import step
from sagemaker.workflow.parameters import ParameterString

# Set path to config file
os.environ["SAGEMAKER_USER_CONFIG_OVERRIDE"] = os.getcwd()

session = boto3.Session(region_name='us-east-1', profile_name='sagemaker')
sagemaker_session = Session(boto_session=session)
s3 = session.client('s3')
#sm = Session()
#s3 = sm.boto_session.client('s3')
region = sagemaker_session.boto_region_name
#role_arn = os.environ.get('SAGEMAKER_EXECUTION_ROLE_ARN')
role = get_execution_role(sagemaker_session=sagemaker_session)
print(f"Using role: {role}")

# sagemaker_session = sagemaker.session.Session()
# region = sagemaker_session.boto_region_name
#role = sagemaker.get_execution_role()
#pipeline_session = PipelineSession()
#default_bucket = sagemaker_session.default_bucket()
#model_package_group_name = f"IrrigationModelPackageGroupName"




INFO:botocore.credentials:Found credentials in environment variables.


Using role: arn:aws:iam::801638386573:role/service-role/AmazonSageMaker-ExecutionRole-20260227T135383


In [28]:
# Location of our dataset
bucket_name = 'jrm-kaggle'
prefix = 'playgrounds6ep4/'
object_name = 'train.csv'
input_path = f"s3://{bucket_name}/{prefix}{object_name}"

pipeline_name = "lowcode-irrigation-xgb"
model_package_group_name = "lowcode-irrigation-xgb"

instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")
model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval"
)
random_state = 2023
label_column = "Irrigation_Need"



In [19]:




@step(
    name="data-preprocessing",
    instance_type=instance_type,
    keep_alive_period_in_seconds=300,
)
def preprocess(raw_data_s3_path: str, output_prefix: str, testing_mode=True, balanced=True) -> tuple:
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    import numpy as np
    from sklearn.utils import resample
    from sklearn.model_selection import train_test_split
    df_tv = pd.read_csv(raw_data_s3_path)
    df_x = df_tv.iloc[:,1:-1]

    df_dummy = pd.get_dummies(df_x, dtype=int, drop_first=False)
    newcolumns = df_dummy.columns.values.tolist() + ['Irrigation_Need']
    continous_variables = df_dummy.select_dtypes(['float64']).columns
    index = [df_dummy.columns.get_loc(col) for col in continous_variables]

    x = df_dummy.iloc[:,:].values
    y = df_tv.iloc[:,-1].values

    class_le = LabelEncoder()
    y = class_le.fit_transform(y)

    if testing_mode:
        X_train, X_test, y_train, y_test = \
            train_test_split(x, y, 
                            test_size=0.20,
                            stratify=y,
                            random_state=1)
    else:
        X_train, y_train = x, y

    sc = StandardScaler().fit(X_train[:, index])
    X_train[:, index] = sc.transform(X_train[:, index])

    if testing_mode:
        X_test[:, index] = sc.transform(X_test[:, index])


    if balanced:
        majority_class = np.argmax(np.bincount(y_train))
        minority_class = np.argmin(np.bincount(y_train))
        middle_class = list(set(np.unique(y_train)) - set([majority_class, minority_class]))[0]
        X_train_majority = X_train[y_train == majority_class]
        y_train_majority = y_train[y_train == majority_class]
        
        X_train_minority = X_train[y_train == minority_class]
        y_train_minority = y_train[y_train == minority_class]
        
        X_train_middle = X_train[y_train == middle_class]
        y_train_middle = y_train[y_train == middle_class]
        
        
        X_train_minority_upsampled, y_train_minority_upsampled = resample(X_train_minority, y_train_minority,
                                                                        replace=True,
                                                                        n_samples=X_train_middle.shape[0],
                                                                        random_state=1)
        X_train_majority_downsampled, y_train_majority_downsampled = resample(X_train_majority, y_train_majority,
                                                                        replace=False,
                                                                        n_samples=X_train_middle.shape[0],
                                                                        random_state=1)
        X_train_balanced = np.vstack((X_train_majority_downsampled, X_train_middle, X_train_minority_upsampled))
        y_train_balanced = np.hstack((y_train_majority_downsampled, y_train_middle, y_train_minority_upsampled))

        perm = np.random.permutation(len(X_train_balanced))

        X_train = X_train_balanced[perm]
        y_train = y_train_balanced[perm]

        train = np.hstack((X_train, y_train.reshape(-1, 1)))
        test = np.hstack((X_test, y_test.reshape(-1, 1)))
        #df_train = pd.DataFrame(train_features, columns=newcolumns)
        train_s3_path = f"s3://{bucket_name}/{output_prefix}/train.csv"
        #val_s3_path = f"s3://{bucket_name}/{output_prefix}/val.csv"
        test_s3_path = f"s3://{bucket_name}/{output_prefix}/test.csv"
        train_df = pd.DataFrame(train, columns=newcolumns)
        test_df = pd.DataFrame(test, columns=newcolumns)
        train_df.to_csv(train_s3_path, index=False)
        #validation_df.to_csv(val_s3_path, index=False)
        test_df.to_csv(test_s3_path, index=False)

        #np.savetxt(train_s3_path, train, delimiter=",")
        #np.savetxt(test_s3_path, test, delimiter=",")
        return train_s3_path, test_s3_path

In [20]:
@step(
    name="model-training",
    instance_type=instance_type,
    keep_alive_period_in_seconds=300,
)
def train(train_s3_path: str):
    from xgboost import XGBClassifier
    import pandas as pd
    # read data files from S3
    train_df = pd.read_csv(train_s3_path)
    X_train = train_df.iloc[:,:-1].values
    y_train = train_df.iloc[:,-1].values
    xgb = XGBClassifier(random_state=1, tree_method='hist', device='cuda', n_jobs=-1,
                       n_estimators=2000, learning_rate=0.1, max_depth=10, reg_lambda=1, 
                       objective='multi:softmax', num_class=3, eval_metric='merror')
    xgb.fit(X_train, y_train)
    return xgb

In [21]:
@step(
    name="model-evaluation",
    instance_type=instance_type,
    keep_alive_period_in_seconds=300,
)
def evaluate(model, test_s3_path: str) -> dict:
    import json
    import numpy as np
    import pandas as pd
    from sklearn.metrics import (
        accuracy_score,
        auc,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_curve,
        classification_report,
        roc_auc_score,
    )
    test_df = pd.read_csv(test_s3_path)
    X_test = test_df.iloc[:,:-1].values
    y_test = test_df.iloc[:,-1].values
    predictions = model.predict_proba(X_test)

    print("Creating classification evaluation report")
    report_dict = classification_report(y_test, predictions.argmax(axis=1), output_dict=True)
    report_dict["accuracy"] = accuracy_score(y_test, predictions.argmax(axis=1))
    #report_dict["roc_auc"] = roc_auc_score(y_test, predictions, multi_class="ovo")

    print("Classification report:\n{}".format(report_dict))
    return report_dict

In [22]:
@step(
    name="model-registration",
    instance_type=instance_type,
    keep_alive_period_in_seconds=300,
)
def register(model, evaluation, model_approval_status, sample_data):
    import json
    import numpy as np
    import pandas as pd
    import s3fs
    from pathlib import Path
    from sagemaker import MetricsSource, ModelMetrics
    from sagemaker.serve.builder.model_builder import ModelBuilder
    from sagemaker.serve.builder.schema_builder import SchemaBuilder
    from sagemaker.serve.spec.inference_spec import InferenceSpec
    from sagemaker.utils import unique_name_from_base
    from xgboost import XGBClassifier

    class XGBoostSpec(InferenceSpec):
        def load(self, model_dir: str):
            print(model_dir)
            model = XGBClassifier()
            model.load_model(model_dir + "/xgboost-model")
            return model

        def invoke(self, input_object: object, model: object):
            prediction_probabilities = model.predict_proba(input_object)
            predictions = np.argmax(prediction_probabilities, axis=1)
            return predictions

    # Upload evaluation report to s3
    eval_file_name = unique_name_from_base("evaluation")
    eval_report_s3_uri = (
        f"s3://{bucket_name}/{model_package_group_name}/evaluation-report/{eval_file_name}.json"
    )
    s3_fs = s3fs.S3FileSystem()
    eval_report_str = json.dumps(evaluation)
    with s3_fs.open(eval_report_s3_uri, "wb") as file:
        file.write(eval_report_str.encode("utf-8"))

    # Create model_metrics as per evaluation report in s3
    model_metrics = ModelMetrics(
        model_statistics=MetricsSource(
            s3_uri=eval_report_s3_uri,
            content_type="application/json",
        )
    )

    sample_data = pd.read_csv(sample_data, nrows=10)
    sample_data.pop(label_column)

    schema_builder = SchemaBuilder(
        sample_input=sample_data.to_numpy(),
        sample_output=model.predict(sample_data),
    )

    model_path = Path("/tmp/model/")
    model_path.mkdir(parents=True, exist_ok=True)
    model.save_model(model_path / "xgboost-model")

    from sagemaker import image_uris
    sklearn_image_uri = image_uris.retrieve(
        framework="sklearn",
        region=region,
        version="1.4-2",  # Use appropriate version
        py_version="py3",
        instance_type="ml.m5.large"
    )

    # Build the trained model and register it
    model_builder = ModelBuilder(
        model_path=str(model_path),
        inference_spec=XGBoostSpec(),
        schema_builder=schema_builder,
        image_uri=sklearn_image_uri,
        #sage_maker_session=sagemaker_session,
        #model=model,
        role_arn=role,
        s3_model_data_url=f"s3://{bucket_name}/{model_package_group_name}/model-artifacts",
        dependencies={"auto":True}
    )
    model_package = model_builder.build().register(
        model_package_group_name=model_package_group_name,
        approval_status=model_approval_status,
        model_metrics=model_metrics,
    )

    print(f"Registered Model Package ARN: {model_package.model_package_arn}")
    return model_package.model_package_arn

In [23]:
from sagemaker.workflow.pipeline import Pipeline

delayed_data = preprocess(
    raw_data_s3_path=input_path,
    output_prefix=f"{prefix}{pipeline_name}/dataset",
)
delayed_model = train(train_s3_path=delayed_data[0])
delayed_evaluation = evaluate(model=delayed_model, test_s3_path=delayed_data[1])
delayed_register = register(
    model=delayed_model,
    evaluation=delayed_evaluation,
    model_approval_status=model_approval_status,
    sample_data=delayed_data[1],
)

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        instance_type,
        model_approval_status,
    ],
    steps=[
        delayed_register,
    ],
    sagemaker_session=sagemaker_session,
)

In [24]:
pipeline.upsert(role_arn=role)

execution = pipeline.start()

execution.describe()

execution.wait()

execution.list_steps()



INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns


2026-04-09 22:16:47,863 sagemaker.remote_function INFO     Uploading serialized function code to s3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb/model-registration/2026-04-09-22-16-44-242/function
2026-04-09 22:16:48,187 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb/model-registration/2026-04-09-22-16-44-242/arguments
2026-04-09 22:16:48,922 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmpoq3l9tmn/requirements.txt'
2026-04-09 22:16:49,016 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb/model-registration/2026-04-09-22-16-44-242/pre_exec_script_and_dependencies'
2026-04-09 22:16:50,661 sagemaker.remote_function INFO     Copied user workspace to '/tmp/tmp37qrg95f/temp_workspace/sagemaker_remote_function_workspace'
2026-04-09 

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns


2026-04-09 22:17:07,558 sagemaker.remote_function INFO     Uploading serialized function code to s3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb/model-training/2026-04-09-22-16-44-242/function
2026-04-09 22:17:07,745 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb/model-training/2026-04-09-22-16-44-242/arguments
2026-04-09 22:17:07,957 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmp6j17orit/requirements.txt'
2026-04-09 22:17:08,051 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb/model-training/2026-04-09-22-16-44-242/pre_exec_script_and_dependencies'


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns


2026-04-09 22:17:10,787 sagemaker.remote_function INFO     Uploading serialized function code to s3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb/model-evaluation/2026-04-09-22-16-44-242/function
2026-04-09 22:17:10,980 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb/model-evaluation/2026-04-09-22-16-44-242/arguments
2026-04-09 22:17:11,185 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmpbjraceba/requirements.txt'
2026-04-09 22:17:11,277 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb/model-evaluation/2026-04-09-22-16-44-242/pre_exec_script_and_dependencies'


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns


2026-04-09 22:17:13,998 sagemaker.remote_function INFO     Uploading serialized function code to s3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb/data-preprocessing/2026-04-09-22-16-44-242/function
2026-04-09 22:17:14,227 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb/data-preprocessing/2026-04-09-22-16-44-242/arguments
2026-04-09 22:17:14,439 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmptc0vb3a6/requirements.txt'
2026-04-09 22:17:14,531 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb/data-preprocessing/2026-04-09-22-16-44-242/pre_exec_script_and_dependencies'
2026-04-09 22:17:15,514 sagemaker.remote_function INFO     Uploading serialized function code to s3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb/model-r

[{'StepName': 'model-registration',
  'StepDisplayName': '__main__.register',
  'StartTime': datetime.datetime(2026, 4, 9, 18, 31, 2, 836000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 4, 9, 18, 34, 11, 580000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'TrainingJob': {'Arn': 'arn:aws:sagemaker:us-east-1:801638386573:training-job/pipelines-9zataj70cxit-model-registration-oMuXEVEpid'}},
  'AttemptCount': 1},
 {'StepName': 'model-evaluation',
  'StepDisplayName': '__main__.evaluate',
  'StartTime': datetime.datetime(2026, 4, 9, 18, 28, 21, 805000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 4, 9, 18, 31, 2, 267000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'TrainingJob': {'Arn': 'arn:aws:sagemaker:us-east-1:801638386573:training-job/pipelines-9zataj70cxit-model-evaluation-JA0StSgczi'}},
  'AttemptCount': 1},
 {'StepName': 'model-training',
  'StepDisplayName': '__main__.train',
  'StartTime': datetime.datetime(2026, 4, 9, 

In [38]:
import boto3
sm = boto3.client('sagemaker')
# Get the latest model
#model_package_arn = execution.result(step_name="model-registration")
model_package_arn = 'arn:aws:sagemaker:us-east-1:801638386573:model-package/lowcode-irrigation-xgb/1'
model_package_update_input_dict = {
    "ModelPackageArn": model_package_arn,
    "ModelApprovalStatus": "Approved",
}
model_package_update_response = sm.update_model_package(**model_package_update_input_dict)

In [39]:

sm.describe_model_package(ModelPackageName=model_package_arn)

model_package = sm.describe_model_package(ModelPackageName=model_package_arn)
model_artifact_s3_path = model_package["InferenceSpecification"]["Containers"][0]["ModelDataUrl"]

In [47]:
import pandas as pd

# Location of our dataset
bucket_name = 'jrm-kaggle'
prefix = 'playgrounds6ep4/'
object_name = 'test_processed.csv'
batch_dataset_s3_path = f"s3://{bucket_name}/{prefix}{object_name}"

# data_df = pd.read_csv(input_path)

# batch_dataset_filename = "data/irrigation_to_be_predicted.csv"

# data_df.to_csv(batch_dataset_filename, index=False)
# batch_dataset_s3_path = sagemaker_session.upload_data(
#     path=batch_dataset_filename,
#     key_prefix=f"{pipeline_name}/dataset/",
# )

In [48]:
inference_pipeline_name = "lowcode-irrigation-xgb-inference"
@step(
    name="batch-inference",
    instance_type=instance_type,
    keep_alive_period_in_seconds=300,
)
def batch_inference(
    model_package_s3_path: str,
    dataset_s3_path: str,
    output_s3_path: str,
):
    import numpy as np
    import pandas as pd
    import s3fs
    import tarfile
    from xgboost import XGBClassifier

    package_filename = "serve.tar.gz"
    s3_fs = s3fs.S3FileSystem()
    s3_fs.get_file(model_package_s3_path, package_filename)
    with tarfile.open(package_filename) as tar:
        tar.extractall(path=".")
    model = XGBClassifier()
    model.load_model("xgboost-model")

    df = pd.read_csv(dataset_s3_path)
    prediction_probabilities = model.predict_proba(df)
    predictions = np.argmax(prediction_probabilities, axis=1)
    with s3_fs.open(output_s3_path, "wb") as file:
        file.write("\n".join(np.char.mod("%d", predictions)).encode("utf-8"))



In [49]:
from sagemaker.workflow.pipeline import Pipeline

delayed_inference = batch_inference(
    model_artifact_s3_path,
    batch_dataset_s3_path,
    f"s3://{bucket_name}/{inference_pipeline_name}/output/predictions.txt",
)

batch_inference_pipeline = Pipeline(
    name=inference_pipeline_name,
    parameters=[
        instance_type,
    ],
    steps=[
        delayed_inference,
    ],
    sagemaker_session=sagemaker_session,
)

In [51]:
batch_inference_pipeline.upsert(role_arn=role)

execution = batch_inference_pipeline.start()

execution.describe()

execution.wait()

execution.list_steps()



INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
2026-04-10 14:32:58,818 sagemaker.remote_function INFO     Uploading serialized function code to s3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb-inference/batch-inference/2026-04-10-14-32-58-817/function
2026-04-10 14:32:59,227 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://sagemaker-us-east-1-801638386573/lowcode-irrigation-xgb-inference/batch-inference/2026-04-10-14-32-58-817/arguments
2026-04-10 14:32:59,998 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmp8_h2j5r4/req

[{'StepName': 'batch-inference',
  'StartTime': datetime.datetime(2026, 4, 10, 10, 33, 35, 199000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 4, 10, 10, 36, 34, 962000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'TrainingJob': {'Arn': 'arn:aws:sagemaker:us-east-1:801638386573:training-job/pipelines-fdjwwhbr7fvg-batch-inference-OAqo3M9Ngc'}},
  'AttemptCount': 1}]